## PRACTICA OBLIGATORIA: **Introducción Deep Learning**

* La práctica obligatoria de esta unidad consiste en un único ejercicio de modelado del dataset del titanic empleando y comparando dos modelos diferentes. Descarga este notebook en tu ordenador y trabaja en local. Ten en cuenta que tendrás que descar los directorios de imágenes y datos adicionales, si los hubiera.
* Recuerda que debes subirla a tu repositorio personal antes de la sesión en vivo para que puntúe adecuadamente.  
* Recuerda también que no es necesario que esté perfecta, sólo es necesario que se vea el esfuerzo. 
* Esta práctica se resolverá en la sesión en vivo correspondiente y la solución se publicará en el repo del curso. 

### Ejercicio 0

Importa los paquetes y módulos que necesites a lo largo del notebook.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report, confusion_matrix

import time


### Descripción y objetivo

El objetivo de la práctica es crear un modelo DL y compararlo con un modelo Random Forest para el dataset de titanic los dos con el mismo objetivo, predecir la supervivencia de un pasajero.  Se pide:  
1. Desarrollar el proceso de ML hasta crear los dos modelos DL y Random Forest. El primero debe tener una topología MLP (es decir una red densa) con un máximo de 3 capas ocultas y debes emplear sklearn para crearlo. No es necesario hacer una análisis/seleccion exahustivo. Escoge una métrica acorde al tipo de target del problema. 

2. Ambos modelos deben tener sus hiperparámetros optimizados, mediante GridSearch. Para ello: utiliza el grid de parámetros que creas conveniente para Random Forest y para el modelo DL, utiliza un grid con los hiperparámetros siguientes:'hidden_layer_sizes','activation', 'solver','alpha' y 'learning_rate'. Para los rangos del grid del modelo de DL guíate por lo visto en el workout y por los posibles valores que se indican en la [documentación](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html#sklearn.neural_network.MLPClassifier). Prueba por lo menos una topología con una sola capa oculta y otra con más de una capa oculta.   
NOTA: Incluye los valores por defecto de los hiperparámetros escogidos en cada caso dentro del grid de hiperparámetros. 
  

3. Compara los modelos respecto a sus métricas medias de accuracy, precision, recall y tiempos de entrenamiento (para ello tendrás que realizar un entrenamiento a parte del mejor modelo obtenido en la optimización de hiperparámetros) y decide cuál te quedarías argumentándolo.    

4. Para el mejor modelo DL obtenido, muestra su clasification report, y la matriz de confusión comentando el resultado.


## 1. Carga de datos

In [2]:
df = pd.read_csv("./data/titanic.csv")
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Limpieza básica

Eliminamos variables redundantes o con demasiados nulos:
- `alive` (equivalente a survived)
- `deck` (muchísimos nulos)
- `class` (duplicado de pclass)


In [3]:
df = df.drop(columns=["alive","deck","class"])
df.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,embark_town,alone
0,0,3,male,22.0,1,0,7.2500,S,man,True,Southampton,False
1,1,1,female,38.0,1,0,71.2833,C,woman,False,Cherbourg,False
2,1,3,female,26.0,0,0,7.9250,S,woman,False,Southampton,True
3,1,1,female,35.0,1,0,53.1000,S,woman,False,Southampton,False
4,0,3,male,35.0,0,0,8.0500,S,man,True,Southampton,True


## 3. Separación de variables

In [4]:
target = "survived"

X = df.drop(columns=[target])
y = df[target]


## 4. Train / Test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


## 5. Preprocesado

- Numéricas → imputación simple + escalado  
- Categóricas → OneHotEncoding


In [6]:
numeric_features = X.select_dtypes(include=["int64","float64","bool"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

from sklearn.impute import SimpleImputer

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


# 6. Modelo Random Forest + GridSearch

In [7]:
rf = RandomForestClassifier(random_state=42)

rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", rf)
])

rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

rf_best = rf_grid.best_estimator_
rf_grid.best_params_


{'model__max_depth': 10,
 'model__min_samples_split': 5,
 'model__n_estimators': 100}

# 7. Modelo Deep Learning (MLPClassifier) + GridSearch

In [8]:
mlp = MLPClassifier(max_iter=1000, random_state=42)

mlp_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", mlp)
])

mlp_param_grid = {
    "model__hidden_layer_sizes": [(50,), (100,), (50,50), (100,50)],
    "model__activation": ["relu","tanh"],
    "model__solver": ["adam","sgd"],
    "model__alpha": [0.0001, 0.001],
    "model__learning_rate": ["constant","adaptive"]
}

mlp_grid = GridSearchCV(
    mlp_pipeline,
    mlp_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

mlp_grid.fit(X_train, y_train)

mlp_best = mlp_grid.best_estimator_
mlp_grid.best_params_


{'model__activation': 'relu',
 'model__alpha': 0.001,
 'model__hidden_layer_sizes': (50,),
 'model__learning_rate': 'constant',
 'model__solver': 'adam'}

# 8. Evaluación y comparación de modelos

In [9]:
def evaluar_modelo(modelo, X_train, X_test, y_train, y_test):

    inicio = time.time()
    modelo.fit(X_train, y_train)
    tiempo = time.time() - inicio

    pred = modelo.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)

    return accuracy, precision, recall, tiempo


In [10]:
rf_acc, rf_prec, rf_rec, rf_time = evaluar_modelo(
    rf_best, X_train, X_test, y_train, y_test
)

mlp_acc, mlp_prec, mlp_rec, mlp_time = evaluar_modelo(
    mlp_best, X_train, X_test, y_train, y_test
)

resultados = pd.DataFrame({
    "Modelo":["Random Forest","MLP"],
    "Accuracy":[rf_acc, mlp_acc],
    "Precision":[rf_prec, mlp_prec],
    "Recall":[rf_rec, mlp_rec],
    "Tiempo entrenamiento":[rf_time, mlp_time]
})

resultados


,Modelo,Accuracy,Precision,Recall,Tiempo entrenamiento
0,Random Forest,0.832402,0.809524,0.739130,0.073913
1,MLP,0.776536,0.730159,0.666667,0.539650


## 9. Conclusión

Se comparan los modelos según:

- Accuracy
- Precision
- Recall
- Tiempo de entrenamiento

En general, **Random Forest suele rendir mejor en datasets tabulares pequeños**, mientras que las redes neuronales necesitan más datos para mostrar ventajas claras.


# 10. Análisis del mejor modelo DL

In [11]:
y_pred_mlp = mlp_best.predict(X_test)

print(classification_report(y_test, y_pred_mlp))


              precision    recall  f1-score   support

           0       0.80      0.85      0.82       110
           1       0.73      0.67      0.70        69

    accuracy                           0.78       179
   macro avg       0.77      0.76      0.76       179
weighted avg       0.77      0.78      0.77       179



In [12]:
confusion_matrix(y_test, y_pred_mlp)


array([[93, 17],
       [23, 46]])

### Interpretación

La matriz de confusión permite observar:

- **True Positives**: supervivientes correctamente predichos
- **True Negatives**: no supervivientes correctamente predichos
- **False Positives**: predice supervivencia cuando no sobrevivió
- **False Negatives**: predice muerte cuando sobrevivió

Esto ayuda a entender el tipo de errores que comete el modelo.
